# Spark SQL Window Functions
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ajit-ai/Data_Science/blob/main/07_BigData_Spark/spark_sql_window_functions.ipynb)

Window functions compute aggregates ACROSS related rows without collapsing them - rankings, running totals, period-over-period deltas. The bread and butter of analytics engineering.

One dataset, both APIs: DataFrame syntax and pure SQL.

In [ ]:
!pip install -q pyspark

## 1. Sales data

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F, Window as W

spark = SparkSession.builder.master("local[*]").appName("windows").getOrCreate()

rows = [("north","alice", 900), ("north","bob",   700), ("north","carol", 700),
        ("south","dave",  650), ("south","erin",  500), ("west", "frank", 800),
        ("west", "grace", 400), ("east", "heidi", 950)]
sales = spark.createDataFrame(rows, "region string, rep string, amount int")
sales.orderBy("region", F.desc("amount")).show()

## 2. Rankings: row_number vs rank vs dense_rank

In [ ]:
w = W.partitionBy("region").orderBy(F.desc("amount"))

sales.withColumn("row_number", F.row_number().over(w)) \
     .withColumn("rank",        F.rank().over(w)) \
     .withColumn("dense_rank",  F.dense_rank().over(w)) \
     .orderBy("region", F.desc("amount")).show()

print("note: bob & carol tie -> rank skips 3, dense_rank does not")

## 3. Running totals + share of region

In [ ]:
run = W.partitionBy("region").orderBy("rep") \
       .rowsBetween(W.unboundedPreceding, W.currentRow)

sales.withColumn("running_total", F.sum("amount").over(run)) \
     .withColumn("region_total",  F.sum("amount").over(W.partitionBy("region"))) \
     .withColumn("pct_of_region", F.round(F.col("amount") /
                  F.sum("amount").over(W.partitionBy("region")) * 100, 1)) \
     .orderBy("region", "rep").show()

## 4. lag/lead - period-over-period change

In [ ]:
months = [(r, m, 100 + r*10 + m*7) for r in range(2) for m in range(1, 7)]
rev = spark.createDataFrame(months, "region int, month int, revenue int")

w_t = W.partitionBy("region").orderBy("month")
rev.withColumn("prev_month", F.lag("revenue", 1).over(w_t)) \
   .withColumn("mom_growth", F.round(
        (F.col("revenue") - F.lag("revenue", 1).over(w_t)) /
         F.lag("revenue", 1).over(w_t) * 100, 1)) \
   .orderBy("region", "month").show()

## 5. Same queries in pure SQL

In [ ]:
sales.createOrReplaceTempView("sales")

spark.sql("""
    SELECT region, rep, amount,
           ROW_NUMBER() OVER (PARTITION BY region ORDER BY amount DESC) AS rn,
           ROUND(amount * 100.0 /
                 SUM(amount) OVER (PARTITION BY region), 1)           AS pct_region,
           LAG(amount) OVER (PARTITION BY region ORDER BY rep)        AS prev_rep_amt
    FROM sales
""").show()

## Cheat sheet
| Need | Function |
|---|---|
| top-N per group | `row_number()` filter rn <= N |
| ties handled | `rank()` / `dense_rank()` |
| cumulative KPIs | `sum() over rowsBetween(unboundedPreceding, currentRow)` |
| deltas | `lag()` / `lead()` |
| first/last in group | `first_value()` / `last_value()` |

Pitfall: sorting AFTER a window function needs a fresh `.orderBy()` - window ordering is internal only.